In [ ]:
!pip install tiktoken

  Using cached tiktoken-0.12.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
Using cached tiktoken-0.12.0-cp310-cp310-manylinux_2_28_x86_64.whl (1.2 MB)


In [ ]:
%env CUDA_LAUNCH_BLOCKING=1
%env TORCH_USE_CUDA_DSA=1

env: CUDA_LAUNCH_BLOCKING=1
env: TORCH_USE_CUDA_DSA=1


#Preprocessing into Tokens
* LLM Books that i downloaded 8 books from the Gutenberg Free E Books
* Now here Removing extra space Unnecessary symbols and maintain the simple text here
* then we will convert into tokens here using the TikToken Library
* save the Final Tokens into .npy file so that I can use it for the GPT Model and the LLM DIffusion Model

In [ ]:
import os
import re
import tiktoken
import numpy as np

BASE_DIR = "jl_fs/llm_vs_diffusion_text/LLM_Books"

INPUT_DIR = BASE_DIR
OUTPUT_TEXT = os.path.join(BASE_DIR, "final_books.txt")
OUTPUT_TOKENS = os.path.join(BASE_DIR, "tokens.npy")
os.makedirs(BASE_DIR, exist_ok=True)
SEPARATOR = "\n<|book_sep|>\n"

def clean_gutenberg(text):
    """
    Removes Project Gutenberg header/footer
    and normalizes text
    """

    # Remove header
    start_match = re.search(r"\*\*\* START OF.*?\*\*\*", text)
    if start_match:
        text = text[start_match.end():]

    # Remove footer
    end_match = re.search(r"\*\*\* END OF.*?\*\*\*", text)
    if end_match:
        text = text[:end_match.start()]

    # Normalize unicode quotes/dashes
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("—", "-").replace("–", "-")

    # Remove carriage returns
    text = text.replace("\r", "")

    # Remove excessive empty lines
    lines = text.split("\n")
    lines = [line.strip() for line in lines if line.strip() != ""]

    text = "\n".join(lines)

    return text.strip()

def load_and_clean_books(input_dir):
    books = []

    files = sorted(os.listdir(input_dir))

    for file in files:
        if not file.endswith(".txt"):
            continue

        path = os.path.join(input_dir, file)

        print(f"Processing: {file}")

        with open(path, "r", encoding="utf-8") as f:
            raw = f.read()

        cleaned = clean_gutenberg(raw)

        print(f"  Length after cleaning: {len(cleaned)} chars")

        books.append(cleaned)

    return books

def main():
    print("\n🔷 Loading and cleaning books...\n")

    books = load_and_clean_books(INPUT_DIR)

    print(f"\nTotal books loaded: {len(books)}")


    final_text = SEPARATOR.join(books)


    with open(OUTPUT_TEXT, "w", encoding="utf-8") as f:
        f.write(final_text)

    print(f"Saved merged text → {OUTPUT_TEXT}")


    print("\n🔷 Tokenizing using GPT-2 tokenizer...\n")

    enc = tiktoken.get_encoding("gpt2")

    tokens = enc.encode(final_text)

    tokens = np.array(tokens, dtype=np.int32)

    print(f"Total tokens: {len(tokens)}")
    print(f"Vocab size: {enc.n_vocab}")

    # Save tokens (VERY IMPORTANT for fast training)
    np.save(OUTPUT_TOKENS, tokens)

    print(f"Saved tokens → {OUTPUT_TOKENS}")

    # ==============================
    # SANITY CHECK
    # ==============================
    print("\n🔷 Sanity Check\n")

    print("First 500 characters:\n")
    print(final_text[:500])

    print("\n---\n")

    print("Last 500 characters:\n")
    print(final_text[-500:])

In [ ]:
 main()


🔷 Loading and cleaning books...

Processing: 1.txt
  Length after cleaning: 143416 chars
Processing: 10.txt
  Length after cleaning: 372842 chars
Processing: 11.txt
  Length after cleaning: 1215102 chars
Processing: 12.txt
  Length after cleaning: 396759 chars
Processing: 13.txt
  Length after cleaning: 1299499 chars
Processing: 14.txt
  Length after cleaning: 268403 chars
Processing: 15.txt
  Length after cleaning: 386668 chars
Processing: 2.txt
  Length after cleaning: 138404 chars
Processing: 3.txt
  Length after cleaning: 418285 chars
Processing: 4.txt
  Length after cleaning: 5300188 chars
Processing: 5.txt
  Length after cleaning: 1774471 chars
Processing: 6.txt
  Length after cleaning: 1018023 chars
Processing: 7.txt
  Length after cleaning: 722729 chars
Processing: 8.txt
  Length after cleaning: 141104 chars
Processing: 9.txt
  Length after cleaning: 1130901 chars

Total books loaded: 15

🔷 Merging books with separator...

Saved merged text → jl_fs/llm_vs_diffusion_text/LLM_Bo

In [ ]:
import re
def advanced_clean(text):

    # Remove Gutenberg header/footer
    text = re.sub(r"\*\*\* START OF.*?\*\*\*", "", text, flags=re.DOTALL)
    text = re.sub(r"\*\*\* END OF.*?\*\*\*", "", text, flags=re.DOTALL)

    # Remove table of contents
    text = re.sub(r"Contents.*?CHAPTER I\.", "CHAPTER I.", text, flags=re.DOTALL)

    # Remove [Illustration]
    text = re.sub(r"\[.*?\]", "", text)

    # Remove decorative stars
    text = re.sub(r"\*[\s\*]+\*", "", text)

    # Remove underscores formatting
    text = text.replace("_", "")

    # Normalize quotes/dashes
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("—", "-").replace("–", "-")

    # Trim before first chapter
    match = re.search(r"(CHAPTER\s+I\.?|Chapter\s+I)", text)
    if match:
        text = text[match.start():]

    # Clean spacing
    text = text.replace("\r", "")
    lines = text.split("\n")
    lines = [line.strip() for line in lines if line.strip() != ""]
    text = "\n".join(lines)

    return text.strip()

In [ ]:
cleaned_books = []

for book in books:
    cleaned = advanced_clean(book)
    cleaned_books.append(cleaned)

In [ ]:
books_small = books[:8]
print("Using books:", len(books_small))

Using books: 8


In [ ]:
final_text = "\n<|book_sep|>\n".join(cleaned_books)

In [ ]:
text_small = "\n<|book_sep|>\n".join(books_small)

In [ ]:
with open("jl_fs/llm_vs_diffusion_text/LLM_Books/final_books_v2.txt", "w", encoding="utf-8") as f:
    f.write(final_text)

In [ ]:
with open("jl_fs/llm_vs_diffusion_text/LLM_Books/final_books_8.txt", "w", encoding="utf-8") as f:
    f.write(text_small)

In [ ]:
import tiktoken
import numpy as np

enc = tiktoken.get_encoding("gpt2")

with open("jl_fs/llm_vs_diffusion_text/LLM_Books/final_books_8.txt", "r", encoding="utf-8") as f:
    text = f.read()

tokens = enc.encode(text)
tokens = np.array(tokens, dtype=np.int32)

np.save("jl_fs/llm_vs_diffusion_text/LLM_Books/tokens_8.npy", tokens)

print("Total tokens:", len(tokens))

Total tokens: 989476


In [ ]:
import re
import tiktoken
import numpy as np


INPUT_TXT = "jl_fs/llm_vs_diffusion_text/LLM_Books/final_books_8.txt"

CLEAN_TXT = "jl_fs/llm_vs_diffusion_text/LLM_Books/final_books_8_cleaned.txt"

TOKENS_PATH = "jl_fs/llm_vs_diffusion_text/LLM_Books/tokens_8.npy"

# =========================================================
# LOAD RAW TEXT
# =========================================================

with open(INPUT_TXT, "r", encoding="utf-8") as f:
    text = f.read()

print("Original characters:", len(text))

# =========================================================
# NORMALIZE NEWLINES
# =========================================================

text = text.replace("\r\n", "\n")
text = text.replace("\r", "\n")

# =========================================================
# SPLIT BOOKS USING EXISTING SEPARATOR
# =========================================================

books = text.split("<|book_sep|>")

cleaned_books = []

# =========================================================
# CLEAN EACH BOOK
# =========================================================

for book in books:

    book = book.strip()

    if len(book) < 100:
        continue

    # =====================================================
    # REMOVE GUTENBERG HEADERS/FOOTERS
    # =====================================================

    book = re.sub(
        r"(?is)project gutenberg.*?start of.*?\n",
        "",
        book
    )

    book = re.sub(
        r"(?is)end of the project gutenberg.*",
        "",
        book
    )

    # =====================================================
    # REMOVE CONTENTS / TABLE OF CONTENTS
    # =====================================================

    book = re.sub(
        r"(?is)table of contents.*?(chapter|chapter i|part i)",
        "",
        book
    )

    # =====================================================
    # REMOVE CHAPTER TITLES / HEADINGS
    # =====================================================

    lines = book.split("\n")

    cleaned_lines = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        upper = line.upper()

        # ---------------------------------------------
        # REMOVE SHORT TITLES / HEADINGS
        # ---------------------------------------------

        if re.match(r"^chapter\s+[ivxlcdm\d]+", upper):
            continue

        if re.match(r"^book\s+[ivxlcdm\d]+", upper):
            continue

        if re.match(r"^part\s+[ivxlcdm\d]+", upper):
            continue

        if re.match(r"^contents?$", upper):
            continue

        # ---------------------------------------------
        # REMOVE VERY SHORT HEADING LINES
        # ---------------------------------------------

        words = line.split()

        if len(words) <= 5:

            # mostly uppercase = heading
            uppercase_ratio = sum(
                c.isupper() for c in line
            ) / max(len(line), 1)

            if uppercase_ratio > 0.6:
                continue

        cleaned_lines.append(line)

    # =====================================================
    # JOIN INTO CLEAN PARAGRAPH TEXT
    # =====================================================

    book = " ".join(cleaned_lines)

    # =====================================================
    # REMOVE MULTIPLE SPACES
    # =====================================================

    book = re.sub(r"\s+", " ", book)

    book = book.strip()

    # =====================================================
    # KEEP BOOK SEPARATOR
    # =====================================================

    cleaned_books.append(
        book + " <|book_sep|>"
    )

# =========================================================
# FINAL CLEAN TEXT
# =========================================================

final_text = "\n".join(cleaned_books)

# =========================================================
# SAVE CLEANED TXT
# =========================================================

with open(CLEAN_TXT, "w", encoding="utf-8") as f:
    f.write(final_text)

print("Saved cleaned txt:")
print(CLEAN_TXT)

print("\nCleaned characters:", len(final_text))

# =========================================================
# TOKENIZATION
# =========================================================

enc = tiktoken.get_encoding("gpt2")

tokens = enc.encode(final_text)

tokens = np.array(tokens, dtype=np.int32)

# =========================================================
# SAVE TOKENS
# =========================================================

np.save(TOKENS_PATH, tokens)

print("\nSaved token file:")
print(TOKENS_PATH)

print("\nTotal tokens:", len(tokens))

Original characters: 3906832
Saved cleaned txt:
jl_fs/llm_vs_diffusion_text/LLM_Books/final_books_8_cleaned.txt

Cleaned characters: 3857137

Saved token file:
jl_fs/llm_vs_diffusion_text/LLM_Books/tokens_8.npy

Total tokens: 896419


In [ ]:
!pip install imageio

  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
Using cached imageio-2.37.3-py3-none-any.whl (317 kB)


# Autoregressive Model
* Transformer Model like GPT from Scratch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import tiktoken
import imageio

from PIL import Image, ImageDraw
from tqdm import tqdm
import os
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class GPTDataset(torch.utils.data.Dataset):
    def __init__(self, tokens, context_length):
        self.tokens = tokens
        self.context_length = context_length

    def __len__(self):
        return len(self.tokens) - self.context_length

    def __getitem__(self, idx):
        x = self.tokens[idx:idx+self.context_length]
        y = self.tokens[idx+1:idx+self.context_length+1]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)  # (B, T, 3C)
        qkv = qkv.view(B, T, 3, self.num_heads, self.head_dim)

        q, k, v = qkv.unbind(dim=2)

        # (B, heads, T, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Attention
        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device))
        attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)

        out = attn @ v  # (B, heads, T, head_dim)

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.proj(out)

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, context_length):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)

        self.blocks = nn.Sequential(*[
            Block(d_model, num_heads) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(0, T, device=x.device)

        x = self.token_emb(x) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln_f(x)

        return self.head(x)

In [ ]:
def generate(model, x, max_new_tokens=50, temperature=1.0, top_k=50):
    model.eval()

    for _ in range(max_new_tokens):
        logits = model(x)
        logits = logits[:, -1, :] / temperature

        if top_k:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)

        x = torch.cat([x, next_token], dim=1)

    return x

In [ ]:
def generate_gif(model, enc, prompt, epoch, path):
    os.makedirs(path,exist_ok=True)
    x = torch.tensor([enc.encode(prompt)], device=device)

    frames = []

    for _ in range(50):
        x = generate(model, x, 1)
        decoded = enc.decode(x[0].tolist())
        frames.append(text_to_image(decoded))

    imageio.mimsave(f"{path}/step_{epoch}.gif", frames, duration=0.3)

In [ ]:
tokens = np.load("jl_fs/llm_vs_diffusion_text/LLM_Books/tokens_8.npy")
context_length = 256
vocab_size = 50257
token_dim = 512
num_heads = 8
num_layers = 6
lr = 3e-4
dataset = GPTDataset(tokens, context_length)
loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)
model = GPT(vocab_size, token_dim, num_heads, num_layers, context_length).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
enc = tiktoken.get_encoding("gpt2")
global_step = 0
path = 'jl_fs/llm_vs_diffusion_text/LLM_Books/llm_model.pth'

for epoch in range(10):

    model.train()
    total_loss = 0

    pbar = tqdm(loader)

    for i, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 200 == 0:
            pbar.set_description(f"Epoch {epoch} Loss {loss.item():.4f}")
        global_step += 1
        if global_step % 1000 == 0:
            generate_gif(
                model,
                enc,
                "Alice was",
                global_step,   # use step instead of epoch
                "jl_fs/llm_vs_diffusion_text/LLM_Books/training_gif"
            )
            torch.save({
                        "model": model,
                        "optimizer": optimizer,
                        "global_step": global_step,
                        "epoch": epoch
                    }, path)
    avg_loss = total_loss / len(loader)
    print(f"\n🔥 Epoch {epoch} Average Loss: {avg_loss:.4f}\n")

Epoch 0 Loss 0.1183: 100%|██████████| 7002/7002 [1:19:15<00:00,  1.47it/s]



🔥 Epoch 0 Average Loss: 1.0363



Epoch 1 Loss 0.0790: 100%|██████████| 7002/7002 [1:19:18<00:00,  1.47it/s]



🔥 Epoch 1 Average Loss: 0.0947



Epoch 2 Loss 0.0639: 100%|██████████| 7002/7002 [1:19:16<00:00,  1.47it/s]


...


🔥 Epoch 7 Average Loss: 0.0455



Epoch 8 Loss 0.0449: 100%|██████████| 7002/7002 [1:19:20<00:00,  1.47it/s]



🔥 Epoch 8 Average Loss: 0.0438



Epoch 9 Loss 0.0451: 100%|██████████| 7002/7002 [1:19:19<00:00,  1.47it/s]


🔥 Epoch 9 Average Loss: 0.0425



In [ ]:
torch.save({
                        "model": model,
                        "optimizer": optimizer,
                        "global_step": global_step,
                        "epoch": epoch
                    }, path)

In [ ]:
def text_to_image(text):
    from PIL import Image, ImageDraw, ImageFont
    import textwrap

    # Bigger canvas
    img = Image.new('RGB', (1400, 800), color='white')
    draw = ImageDraw.Draw(img)

    # Bigger font
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 48)
    except:
        font = ImageFont.load_default()

    # 🔥 Less characters per line (bigger appearance)
    wrapped = textwrap.fill(text, width=25)

    # 🔥 Show fewer characters → bigger visual text
    display_text = wrapped[-400:]

    draw.text((50, 50), display_text, fill='black', font=font, spacing=10)

    return img

In [ ]:
generate_gif(
                model,
                enc,
                "Alice was",
                   2,   # use step instead of epoch
                "jl_fs/llm_vs_diffusion_text/LLM_Books/inference_gif"
            )

In [ ]:
generate_gif(
                model,
                enc,
                "Once upon",
                   1,   # use step instead of epoch
                "jl_fs/llm_vs_diffusion_text/LLM_Books/inference_gif"
            )

In [ ]:
## LANGUAGE DIFFUSION MODEL

In [ ]:
# @torch.no_grad()
# def diffusion_generate(
#     model,
#     enc,
#     prompt,
#     total_steps=128,
#     temperature=0.7,
#     top_k=20,
#     gen_len=32,
# ):

#     model.eval()

#     # ========================================================
#     # ENCODE PROMPT
#     # ========================================================

#     prompt_ids = enc.encode(prompt)

#     max_prompt_len = context_length - gen_len

#     prompt_ids = prompt_ids[:max_prompt_len]

#     num_prompt_tokens = len(prompt_ids)

#     total_length = num_prompt_tokens + gen_len
#     x = torch.full(
#         (1, total_length),
#         mask_token_id,
#         dtype=torch.long,
#         device=device
#     )

#     x[0, :num_prompt_tokens] = torch.tensor(
#         prompt_ids,
#         dtype=torch.long,
#         device=device
#     )

#     # ========================================================
#     # FIXED TOKENS
#     # ========================================================

#     fixed = torch.zeros(
#         total_length,
#         dtype=torch.bool,
#         device=device
#     )

#     fixed[:num_prompt_tokens] = True
#     frames = []
#     text_frames = []

#     # ========================================================
#     # REVERSE DIFFUSION LOOP
#     # ========================================================

#     for step in range(total_steps, 0, -1):

#         # ====================================================
#         # TIMESTEP
#         # ====================================================

#         t = torch.tensor([step], device=device)
#         logits = model(x, t)

#         # ====================================================
#         # CONTEXTUAL STABILIZATION
#         # IMPORTANT
#         # ====================================================

#         context_strength = 0.15

#         for pos in range(num_prompt_tokens, total_length):

#             left_start = max(0, pos - 8)

#             local_context = x[0, left_start:pos]

#             visible = local_context[
#                 local_context != mask_token_id
#             ]

#             if len(visible) > 0:
#                 avg_emb = model.token_embedding(visible).mean(dim=0)
#                 similarity = torch.matmul(
#                     model.head.weight,
#                     avg_emb
#                 )

#                 logits[0, pos] += (
#                     context_strength * similarity
#                 )
#         current_temp = 0.4 + 0.8 * (
#             step / total_steps
#         )

#         logits = logits / current_temp
#         if top_k and top_k > 0:

#             topk_vals, topk_idx = torch.topk(
#                 logits,
#                 k=top_k,
#                 dim=-1
#             )

#             filtered = torch.full_like(
#                 logits,
#                 float('-inf')
#             )

#             filtered.scatter_(
#                 -1,
#                 topk_idx,
#                 topk_vals
#             )

#             logits = filtered
#         probs = F.softmax(logits, dim=-1)

#         flat = probs.view(-1, probs.size(-1))

#         sampled = torch.multinomial(
#             flat,
#             num_samples=1
#         ).view(1, total_length)

#         # ====================================================
#         # TOKEN CONFIDENCE
#         # ====================================================

#         conf = probs.gather(
#             -1,
#             sampled.unsqueeze(-1)
#         ).squeeze(-1)
#         confidence_threshold = 0.92

#         high_conf_mask = (
#             conf[0] > confidence_threshold
#         ) & (~fixed)

#         fixed[high_conf_mask] = True

#         # ====================================================
#         # ONLY UPDATE MASKED TOKENS
#         # ====================================================

#         update_pos = (
#             x[0] == mask_token_id
#         )

#         x[0, update_pos] = sampled[0, update_pos]

#         # ====================================================
#         # HARD CLAMP PROMPT
#         # ====================================================

#         x[0, :num_prompt_tokens] = torch.tensor(
#             prompt_ids,
#             dtype=torch.long,
#             device=device
#         )
#         next_ratio = math.cos(
#             ((total_steps - step + 1) / total_steps)
#             * math.pi * 0.5
#         )

#         target_masks = int(
#             math.ceil(gen_len * next_ratio)
#         )

#         candidate_positions = torch.where(~fixed)[0]

#         if (
#             target_masks > 0 and
#             candidate_positions.numel() > 0
#         ):

#             candidate_conf = conf[
#                 0,
#                 candidate_positions
#             ]
#             k = min(
#                 target_masks,
#                 candidate_positions.numel()
#             )
#             _, low_idx = torch.topk(
#                 candidate_conf,
#                 k=k,
#                 largest=False
#             )
#             remask_positions = candidate_positions[
#                 low_idx
#             ]
#             x[0, remask_positions] = mask_token_id
#         decoded = decode_with_mask(
#             x[0].tolist(),
#             mask_token_id
#         )
#         text_frames.append(decoded)
#         # ====================================================
#         # BUILD GIF FRAME
#         # ====================================================
#         lines = [
#             "=================== LANGUAGE DIFFUSION ===================",
#             f"(reverse diffusion step {total_steps - step + 1:03d}/{total_steps:03d})",
#             "",
#             "[PROMPT]",
#             prompt,
#             "",
#             "[REVERSE DIFFUSION PROCESS]",
#             decoded,
#         ]
#         frame_text = "\n".join(lines)
#         frame = text_to_image(frame_text)
#         frames.append(frame)
#     # ========================================================
#     # FINAL DECODE
#     # ========================================================
#     final_tokens = []
#     for token in x[0].tolist():
#         if token != mask_token_id:
#             final_tokens.append(token)

#     final_text = enc.decode(final_tokens)
#     model.train()
#     return final_text, frames, text_frames

In [ ]:
# def save_diffusion_gif(frames, path):

#     imageio.mimsave(
#         path,
#         frames,
#         duration=0.08
#     )

#     print("Saved:", path)

In [ ]:
# global_step = 0
# best_loss = 999999
# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0
#     pbar = tqdm(loader)
#     for x in pbar:
#         x = x.to(device)
#         loss = diffusion_loss(model, x)
#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()
#         total_loss += loss.item()
#         global_step += 1
#         pbar.set_description(f"Epoch {epoch} Loss {loss.item():.4f}")
#         if global_step % 1000 == 0:
#             with torch.no_grad():
#                 text, frames, text_frames = diffusion_generate(
#                     model,
#                     enc,
#                     prompt="Alice was beginning to get very tired",
#                     total_steps=128,
#                     temperature=0.7,
#                     top_k=20,
#                     gen_len=32
#                 )

#             save_diffusion_gif(
#                 frames,
#                 f"{GIF_DIR}/step_{global_step}.gif"
#             )
#     avg_loss = total_loss / len(loader)
#     print(f"\nEpoch {epoch} Avg Loss: {avg_loss:.4f}\n")

#     # SAVE CHECKPOINT
#     checkpoint = {
#         "model": model.state_dict(),
#         "optimizer": optimizer.state_dict(),
#         "epoch": epoch,
#         "global_step": global_step
#     }
#     torch.save(checkpoint, f"{CHECKPOINT_DIR}/epoch_{epoch}.pt")
#     if avg_loss < best_loss:
#         best_loss = avg_loss
#         torch.save(checkpoint, f"{CHECKPOINT_DIR}/best_model.pt")
#         print("Best model saved")

Epoch 0 Loss 7.1771:   7%|▋         | 1000/14002 [05:58<16:03:26,  4.45s/it]

Saved: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_1000.gif


Epoch 0 Loss 6.9664:  14%|█▍        | 2000/14002 [11:57<13:46:03,  4.13s/it]

Saved: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_2000.gif


Epoch 0 Loss 6.9969:  21%|██▏       | 3000/14002 [17:55<12:34:06,  4.11s/it]

...

Saved: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_4000.gif


Epoch 0 Loss 6.8374:  36%|███▌      | 5000/14002 [29:52<10:23:02,  4.15s/it]

Saved: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_5000.gif


Epoch 0 Loss 6.7458:  43%|████▎     | 5999/14002 [35:47<47:45,  2.79it/s]   


KeyboardInterrupt: 

In [ ]:
# """
# Masked Diffusion Language Model
# ================================
# Trained on tokens_8.npy (GPT-2 BPE token ids from tiktoken).

# Architecture mirrors the reference code exactly:
#   - DiffusionLMConfig dataclass
#   - DiffusionTransformerLM with nn.TransformerEncoder (pre-LN, no causal mask)
#   - clean diffusion_generate: linear re-mask schedule, no contextual bias,
#     no confidence locking, attention_mask passed through

# Tokenizer note
# --------------
# tokens_8.npy was encoded with tiktoken's GPT-2 BPE (vocab 0-50255).
# We extend the vocab by 2:
#     50256 = PAD  (reuses GPT-2's <|endoftext|> slot)
#     50257 = MASK (new special token, same id as original code)

# During TRAINING the tokenizer is never called — the model reads raw ids
# from the .npy file directly, so training is completely unaffected.

# During INFERENCE (encode prompt → generate → decode output):
#   - The code tries tiktoken first (correct GPT-2 BPE).
#   - tiktoken requires a one-time download of ~500KB vocab files.
#     On a machine with internet access this happens automatically.
#   - If tiktoken is unavailable (no network / sandbox), it falls back to
#     a byte-level approximation that still lets the pipeline run end-to-end.
#     Generated token ids will be correct; only the prompt encoding differs.

# To pre-cache tiktoken vocab on a machine with internet, run once:
#     python -c "import tiktoken; tiktoken.get_encoding('gpt2')"
# """

# import os, math, json
# import numpy as np
# from dataclasses import dataclass
# from tqdm import tqdm

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader

# # =========================================================
# # PATHS  — adjust to your environment
# # =========================================================
# TOKENS_PATH = "jl_fs/llm_vs_diffusion_text/LLM_Books/tokens_8.npy"
# CHECKPOINT_DIR = "jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_checkpoints"
# GIF_DIR = "jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs"

# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# os.makedirs(GIF_DIR, exist_ok=True)

# # =========================================================
# # VOCAB CONSTANTS
# # tokens_8.npy uses GPT-2 BPE ids 0-50255.
# # We add PAD and MASK on top, keeping the same layout as the
# # original code (mask_token_id = 50257).
# # =========================================================
# VOCAB_SIZE = 50258   # 50256 GPT-2 tokens + PAD + MASK
# PAD_ID     = 50256   # <|endoftext|> slot reused as PAD
# MASK_ID    = 50257   # new [MASK] special token
# BOS_ID     = 50256   # GPT-2 uses <|endoftext|> as BOS
# EOS_ID     = 50256

# print(f"VOCAB_SIZE={VOCAB_SIZE} | PAD={PAD_ID} MASK={MASK_ID} BOS={BOS_ID}")

# # =========================================================
# # TOKENIZER
# # Try tiktoken (correct GPT-2 BPE). Fall back gracefully.
# # =========================================================
# def _build_tiktoken_tokenizer():
#     """Returns a tokenizer object with .encode() and .decode() using tiktoken."""
#     import tiktoken
#     enc = tiktoken.get_encoding("gpt2")

#     class TiktokenWrapper:
#         def __init__(self, enc):
#             self._enc = enc
#             self.vocab_size    = VOCAB_SIZE
#             self.mask_token_id = MASK_ID
#             self.pad_token_id  = PAD_ID
#             self.bos_token_id  = BOS_ID
#             self.eos_token_id  = EOS_ID

#         def encode(self, text: str, add_special_tokens: bool = True):
#             ids = self._enc.encode(text)
#             if add_special_tokens:
#                 ids = [BOS_ID] + ids
#             return ids

#         def decode(self, ids) -> str:
#             # Filter out special tokens before decoding
#             clean = [i for i in ids if i not in (MASK_ID, PAD_ID) and i < 50256]
#             text = self._enc.decode(clean)
#             # Re-insert mask markers at original positions
#             parts = []
#             clean_ptr = 0
#             for tok in ids:
#                 if tok == MASK_ID:
#                     parts.append("█")
#                 elif tok == PAD_ID:
#                     pass
#                 else:
#                     parts.append(None)   # placeholder; will be filled from decoded text
#             # Simpler: decode clean ids and append mask count info
#             mask_count = sum(1 for i in ids if i == MASK_ID)
#             if mask_count > 0:
#                 return text + " " + "█" * mask_count
#             return text

#     return TiktokenWrapper(enc)


# def _build_fallback_tokenizer():
#     """
#     Fallback tokenizer that works without network access.
#     encode(): byte-level mapping (each char → its ASCII/unicode byte value mod vocab)
#     decode(): renders ids as readable text where possible, █ for mask tokens.

#     NOTE: encode() output won't match GPT-2 BPE ids — only use for quick testing.
#     The training loop never calls encode/decode; only inference prompts are affected.
#     """
#     class FallbackTokenizer:
#         def __init__(self):
#             self.vocab_size    = VOCAB_SIZE
#             self.mask_token_id = MASK_ID
#             self.pad_token_id  = PAD_ID
#             self.bos_token_id  = BOS_ID
#             self.eos_token_id  = EOS_ID

#         def encode(self, text: str, add_special_tokens: bool = True):
#             # Map bytes to token ids in the safe range (0-50254)
#             ids = [b % (VOCAB_SIZE - 3) for b in text.encode("utf-8")]
#             if add_special_tokens:
#                 ids = [BOS_ID] + ids
#             return ids

#         def decode(self, ids) -> str:
#             parts = []
#             run = []
#             for tok in ids:
#                 if tok == MASK_ID:
#                     if run:
#                         try:
#                             parts.append(bytes(r % 256 for r in run).decode("utf-8", errors="replace"))
#                         except Exception:
#                             parts.append(f"[{','.join(str(r) for r in run)}]")
#                         run = []
#                     parts.append("█")
#                 elif tok in (PAD_ID, BOS_ID):
#                     pass
#                 else:
#                     run.append(tok)
#             if run:
#                 try:
#                     parts.append(bytes(r % 256 for r in run).decode("utf-8", errors="replace"))
#                 except Exception:
#                     parts.append(f"[{','.join(str(r) for r in run)}]")
#             return "".join(parts)

#     return FallbackTokenizer()


# try:
#     tokenizer = _build_tiktoken_tokenizer()
#     print("Tokenizer: tiktoken GPT-2 BPE (correct)")
# except Exception as e:
#     tokenizer = _build_fallback_tokenizer()
#     print(f"Tokenizer: fallback (tiktoken unavailable: {e})")
#     print("  → Training is unaffected. For correct inference, install tiktoken")
#     print("    on a machine with internet: pip install tiktoken")

# # =========================================================
# # CONFIG  (mirrors reference budget_100 profile)
# # =========================================================
# @dataclass
# class DiffusionLMConfig:
#     vocab_size:      int
#     seq_len:         int
#     d_model:         int
#     n_layers:        int
#     n_heads:         int
#     d_ff:            int
#     dropout:         float
#     diffusion_steps: int

# cfg = DiffusionLMConfig(
#     vocab_size      = VOCAB_SIZE,
#     seq_len         = 256,
#     d_model         = 512,
#     n_layers        = 6,
#     n_heads         = 8,
#     d_ff            = 4 * 512,
#     dropout         = 0.1,
#     diffusion_steps = 128,
# )

# BATCH_SIZE  = 64
# LR          = 1e-4
# NUM_EPOCHS  = 10
# device      = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device: {device}")

# # =========================================================
# # DATASET
# # =========================================================
# class DiffusionDataset(Dataset):
#     def __init__(self, tokens, seq_len: int):
#         self.tokens  = tokens
#         self.seq_len = seq_len

#     def __len__(self):
#         return len(self.tokens) - self.seq_len

#     def __getitem__(self, idx):
#         block          = self.tokens[idx : idx + self.seq_len]
#         input_ids      = torch.tensor(block, dtype=torch.long)
#         attention_mask = (input_ids != PAD_ID)
#         return {"input_ids": input_ids, "attention_mask": attention_mask}


# tokens = np.load(TOKENS_PATH).astype(np.int64)
# tokens = np.clip(tokens, 0, VOCAB_SIZE - 1)   # guard against any out-of-range ids
# print(f"Tokens loaded — shape: {tokens.shape} | max id: {tokens.max()}")

# dataset = DiffusionDataset(tokens, cfg.seq_len)
# loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
# print(f"Batches per epoch: {len(loader)}")

# # =========================================================
# # MODEL  (mirrors reference DiffusionTransformerLM exactly)
# # =========================================================
# class DiffusionTransformerLM(nn.Module):
#     def __init__(self, cfg: DiffusionLMConfig):
#         super().__init__()
#         self.cfg = cfg

#         self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
#         self.pos_emb  = nn.Embedding(cfg.seq_len,    cfg.d_model)
#         self.time_emb = nn.Embedding(cfg.diffusion_steps + 1, cfg.d_model)

#         enc_layer = nn.TransformerEncoderLayer(
#             d_model         = cfg.d_model,
#             nhead           = cfg.n_heads,
#             dim_feedforward = cfg.d_ff,
#             dropout         = cfg.dropout,
#             batch_first     = True,
#             activation      = "gelu",
#             norm_first      = True,   # pre-LN: more stable training
#         )
#         self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
#         self.ln_f    = nn.LayerNorm(cfg.d_model)
#         self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

#         # Weight tying: token embedding ↔ lm_head (saves ~25M params, standard practice)
#         self.lm_head.weight = self.tok_emb.weight

#         self.drop = nn.Dropout(cfg.dropout)

#     def forward(self, input_ids, timesteps, attention_mask=None):
#         """
#         input_ids      : [B, L]  integer token ids
#         timesteps      : [B]     integer diffusion step in [1..T]
#         attention_mask : [B, L]  bool, True = real token, False = pad
#         returns logits : [B, L, V]
#         """
#         B, L = input_ids.shape

#         pos   = torch.arange(L, device=input_ids.device).unsqueeze(0)   # [1, L]
#         x     = self.tok_emb(input_ids) + self.pos_emb(pos)
#         t_emb = self.time_emb(timesteps).unsqueeze(1)                    # [B, 1, D]
#         x     = x + t_emb
#         x     = self.drop(x)

#         # nn.TransformerEncoder: src_key_padding_mask True = IGNORE (inverted from HF)
#         src_key_padding_mask = (~attention_mask) if attention_mask is not None else None

#         x      = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
#         x      = self.ln_f(x)
#         logits = self.lm_head(x)   # [B, L, V]
#         return logits


# model     = DiffusionTransformerLM(cfg).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)

# n_params = sum(p.numel() for p in model.parameters())
# print(f"Parameters: {n_params / 1e6:.2f}M")

# # =========================================================
# # MASKING / CORRUPTION  (mirrors reference corrupt_with_mask)
# # =========================================================
# def mask_ratio_schedule(t, T: int):
#     """Linear schedule: ratio = t/T  (same as reference)."""
#     return t.float() / float(T)


# @torch.no_grad()
# def corrupt_with_mask(input_ids, attention_mask, t, T: int):
#     """
#     Randomly mask tokens proportional to timestep t.
#     Protects BOS, EOS, PAD from masking.
#     Returns: (noisy_ids, labels, mask_positions)
#       - noisy_ids: input_ids with masked positions replaced by MASK_ID
#       - labels:    original ids at masked positions, -100 elsewhere
#     """
#     B, L  = input_ids.shape
#     ratio = mask_ratio_schedule(t, T).unsqueeze(1)   # [B, 1]

#     can_mask  = attention_mask.clone()
#     can_mask &= (input_ids != BOS_ID) & (input_ids != EOS_ID) & (input_ids != PAD_ID)

#     rand           = torch.rand((B, L), device=input_ids.device)
#     mask_positions = (rand < ratio) & can_mask

#     noisy               = input_ids.clone()
#     noisy[mask_positions] = MASK_ID

#     labels = torch.full_like(input_ids, -100)
#     labels[mask_positions] = input_ids[mask_positions]

#     return noisy, labels, mask_positions


# def diffusion_loss(model, batch, T: int):
#     input_ids      = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)

#     B = input_ids.size(0)
#     t = torch.randint(1, T + 1, (B,), device=device)

#     noisy_ids, labels, _ = corrupt_with_mask(input_ids, attention_mask, t, T)

#     logits = model(noisy_ids, timesteps=t, attention_mask=attention_mask)
#     loss   = F.cross_entropy(
#         logits.view(-1, logits.size(-1)),
#         labels.view(-1),
#         ignore_index=-100,
#     )
#     return loss

# # =========================================================
# # INFERENCE  (clean port of reference diffusion_generate)
# # =========================================================
# @torch.no_grad()
# def diffusion_generate(
#     model,
#     tokenizer,
#     prompt_text:     str,
#     max_new_tokens:  int   = 128,
#     diffusion_steps: int   = 64,
#     temperature:     float = 1.0,
#     top_k:           int   = 0,
#     record_steps:    bool  = True,
# ):
#     """
#     Reverse diffusion: starts with all generation positions masked,
#     iteratively unmasks tokens, re-masking the lowest-confidence ones
#     according to a linear schedule.

#     Returns: (final_text, frames)
#       - final_text: decoded string of the full sequence
#       - frames:     list of (step, decoded_string) for GIF rendering
#     """
#     model.eval()
#     dev = next(model.parameters()).device

#     prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
#     prompt_ids = torch.tensor(prompt_ids, dtype=torch.long, device=dev).unsqueeze(0)  # [1, Lp]

#     Lp      = prompt_ids.size(1)
#     L       = min(cfg.seq_len, Lp + max_new_tokens)
#     gen_len = L - Lp

#     # Initialise: prompt tokens in place, generation span = [MASK]
#     x = torch.full((1, L), MASK_ID, dtype=torch.long, device=dev)
#     x[:, :Lp] = prompt_ids[:, :Lp]

#     # fixed[pos] = True means that position is locked (never re-masked)
#     fixed         = torch.zeros((1, L), dtype=torch.bool, device=dev)
#     fixed[:, :Lp] = True

#     attention_mask = torch.ones((1, L), dtype=torch.bool, device=dev)

#     frames = []

#     def sample_from_logits(logits):
#         """Apply temperature + top-k, sample, return (sampled, confidence)."""
#         if temperature != 1.0:
#             logits = logits / temperature

#         if top_k and top_k > 0:
#             k = min(top_k, logits.size(-1))   # guard: top_k must not exceed vocab
#             topk_vals, topk_idx = torch.topk(logits, k=k, dim=-1)
#             filtered = torch.full_like(logits, float("-inf"))
#             filtered.scatter_(-1, topk_idx, topk_vals)
#             logits = filtered

#         probs        = F.softmax(logits, dim=-1)
#         flat         = probs.view(-1, probs.size(-1))
#         sampled      = torch.multinomial(flat, num_samples=1).view(1, L)
#         sampled_prob = probs.gather(-1, sampled.unsqueeze(-1)).squeeze(-1)   # [1, L]
#         return sampled, sampled_prob

#     for s in range(diffusion_steps, 0, -1):
#         t      = torch.tensor([s], device=dev, dtype=torch.long)
#         logits = model(x, timesteps=t, attention_mask=attention_mask)

#         sampled, conf = sample_from_logits(logits)

#         # Update all non-fixed (non-prompt) positions with the model's prediction
#         update_pos    = ~fixed
#         x[update_pos] = sampled[update_pos]

#         # Linear re-mask schedule: how many tokens to put back as [MASK]
#         next_ratio   = float(s - 1) / float(diffusion_steps)
#         target_masks = int(math.ceil(gen_len * next_ratio))

#         # Only consider generation positions (not prompt) for re-masking
#         gen_positions = torch.arange(L, device=dev) >= Lp
#         candidates    = gen_positions & (~fixed[0])
#         cand_idx      = torch.where(candidates)[0]

#         if target_masks > 0 and cand_idx.numel() > 0:
#             cand_conf = conf[0, cand_idx]
#             k         = min(target_masks, cand_idx.numel())
#             _, low_idx = torch.topk(cand_conf, k=k, largest=False)
#             remask_positions = cand_idx[low_idx]
#             x[0, remask_positions] = MASK_ID

#         if record_steps:
#             decoded = tokenizer.decode(x[0].tolist())
#             frames.append((s, decoded))

#     final = tokenizer.decode(x[0].tolist())
#     model.train()
#     return final, frames

# # =========================================================
# # GIF RENDERING  (mirrors reference render_terminal_frame)
# # =========================================================
# from PIL import Image, ImageDraw, ImageFont
# import imageio.v2 as imageio


# def get_mono_font(size: int = 20):
#     candidates = [
#         "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
#         "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
#     ]
#     for path in candidates:
#         if os.path.exists(path):
#             return ImageFont.truetype(path, size=size)
#     return ImageFont.load_default()


# def wrap_text_to_width(text: str, max_chars: int = 90):
#     out = []
#     for paragraph in text.split("\n"):
#         paragraph = paragraph.rstrip()
#         if not paragraph:
#             out.append("")
#             continue
#         while len(paragraph) > max_chars:
#             out.append(paragraph[:max_chars])
#             paragraph = paragraph[max_chars:]
#         out.append(paragraph)
#     return out


# def make_chat_lines(user_msg: str, assistant_text: str):
#     lines = ["=" * 79, "", "[You]:", user_msg, "", "[Assistant]:"]

#     # Strip prompt template noise from decoded output
#     for sep in ["<|assistant|>", "<|endoftext|>"]:
#         if sep in assistant_text:
#             assistant_text = assistant_text.split(sep, 1)[1]
#     assistant_text = assistant_text.strip()

#     lines += wrap_text_to_width(assistant_text, max_chars=90)
#     return lines


# def render_terminal_frame(
#     lines,
#     width: int = 1200, height: int = 700,
#     font_size: int = 20, margin: int = 20, line_spacing: int = 6,
# ):
#     img  = Image.new("RGB", (width, height), (10, 10, 10))
#     draw = ImageDraw.Draw(img)
#     font = get_mono_font(font_size)
#     y    = margin
#     for line in lines:
#         draw.text((margin, y), line, font=font, fill=(230, 230, 230))
#         y += font_size + line_spacing
#         if y > height - margin:
#             break
#     return img


# def save_diffusion_gif(frames, prompt_text: str, gif_path: str, total_steps: int):
#     gif_frames = []
#     for s, decoded in frames:
#         lines = make_chat_lines(prompt_text, decoded)
#         lines.insert(2, f"(diffusion step {total_steps - s + 1:03d}/{total_steps:03d})")
#         img = render_terminal_frame(lines)
#         gif_frames.append(np.array(img))
#     imageio.mimsave(gif_path, gif_frames, duration=0.08)
#     print(f"Saved GIF: {gif_path}")

# # =========================================================
# # TRAINING LOOP
# # =========================================================
# SAMPLE_PROMPT = "Alice was beginning to get very tired"
# global_step   = 0
# best_loss     = float("inf")

# for epoch in range(NUM_EPOCHS):
#     model.train()
#     total_loss = 0.0
#     pbar = tqdm(loader, desc=f"Epoch {epoch}")

#     for batch in pbar:
#         loss = diffusion_loss(model, batch, T=cfg.diffusion_steps)
#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()

#         total_loss  += loss.item()
#         global_step += 1
#         pbar.set_postfix(loss=f"{loss.item():.4f}")

#         # ── sample every 1000 steps ──────────────────────
#         if global_step % 1000 == 0:
#             final_text, frames = diffusion_generate(
#                 model           = model,
#                 tokenizer       = tokenizer,
#                 prompt_text     = SAMPLE_PROMPT,
#                 max_new_tokens  = 32,
#                 diffusion_steps = cfg.diffusion_steps,
#                 temperature     = 1.0,
#                 top_k           = 50,
#                 record_steps    = True,
#             )
#             gif_path = os.path.join(GIF_DIR, f"step_{global_step}.gif")
#             save_diffusion_gif(frames, SAMPLE_PROMPT, gif_path, cfg.diffusion_steps)
#             print(f"\nStep {global_step} sample:\n{final_text[:300]}\n")

#     avg_loss = total_loss / len(loader)
#     print(f"\nEpoch {epoch} | avg_loss: {avg_loss:.4f}")

#     # ── checkpoint ───────────────────────────────────────
#     ckpt = {
#         "model":       model.state_dict(),
#         "optimizer":   optimizer.state_dict(),
#         "epoch":       epoch,
#         "global_step": global_step,
#         "cfg":         cfg.__dict__,
#     }
#     torch.save(ckpt, os.path.join(CHECKPOINT_DIR, f"epoch_{epoch}.pt"))

#     if avg_loss < best_loss:
#         best_loss = avg_loss
#         torch.save(ckpt, os.path.join(CHECKPOINT_DIR, "best_model.pt"))
#         print("  ✓ Best model saved")

# # =========================================================
# # FINAL INFERENCE DEMO
# # =========================================================
# print("\n--- Final inference demo ---")

# final_text, frames = diffusion_generate(
#     model           = model,
#     tokenizer       = tokenizer,
#     prompt_text     = SAMPLE_PROMPT,
#     max_new_tokens  = 128,
#     diffusion_steps = cfg.diffusion_steps,
#     temperature     = 1.0,
#     top_k           = 50,
#     record_steps    = True,
# )

# print("Generated text:\n", final_text[:500])
# print(f"Recorded frames: {len(frames)}")

# save_diffusion_gif(
#     frames, SAMPLE_PROMPT,
#     os.path.join(GIF_DIR, "final_inference.gif"),
#     cfg.diffusion_steps,
# )

VOCAB_SIZE=50258 | PAD=50256 MASK=50257 BOS=50256
Tokenizer: tiktoken GPT-2 BPE (correct)
Device: cuda
Tokens loaded — shape: (896419,) | max id: 50255
Batches per epoch: 14002


/tmp/ipykernel_348/265361613.py:252: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)


Parameters: 44.84M


Epoch 0:   7%|▋         | 1000/14002 [05:47<11:33:24,  3.20s/it, loss=10.2397]

Saved GIF: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_1000.gif

Step 1000 sample:
Alice was beginning to get very tired the a as on but,, but is, on to, that; the," the and the other as. the toed this it the the I is



...

Saved GIF: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_139000.gif

Step 139000 sample:
Alice was beginning to get very tired the which nature, M," M man Mught M was M it M and Meking not than, M in M. M or M now M.



Epoch 9: 100%|█████████▉| 13982/14002 [1:21:32<01:00,  3.03s/it, loss=4.0920]

Saved GIF: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/step_140000.gif

Step 140000 sample:
Alice was beginning to get very tired the this curiosity all Dr, J the Jekyll of salt coloured brow through bright brow bright brow of I the open it he as bright swazuding of



Epoch 9: 100%|██████████| 14002/14002 [1:21:39<00:00,  2.86it/s, loss=4.1076]



Epoch 9 | avg_loss: 4.0971
  ✓ Best model saved

--- Final inference demo ---
Generated text:
 Alice was beginning to get very tired of the wife, flattened," wife; air at one- after Mr that Wilde, answer and Wilde " door the I only over curnnnnnn in- Stey at by. ofless by for., there his for wifeless neednnold Mr was Wilde, wifeless over her; wiferils over as with forThen this Chester but Chester, Chester and Chester to Chester to Chester, Chester Chester as Chester Chester Chester only with Chester Chester Chester but Reputation, Chester, Chester Chester Chester, Chester the Chester Ches
Recorded frames: 128
Saved GIF: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/final_inference.gif


In [ ]:
final_text, frames = diffusion_generate(
    model           = model,
    tokenizer       = tokenizer,
    prompt_text     = "Alice was",
    max_new_tokens  = 128,
    diffusion_steps = cfg.diffusion_steps,
    temperature     = 1.0,
    top_k           = 50,
    record_steps    = True,
)

print("Generated text:\n", final_text[:500])
print(f"Recorded frames: {len(frames)}")

save_diffusion_gif(
    frames, SAMPLE_PROMPT,
    os.path.join(GIF_DIR, "final_inference.gif"),
    cfg.diffusion_steps,
)

Generated text:
 Alice was and a you and nature; the Utterson to the to and to name not. in his human misend over; Utterson the loend, human entire proportion in for, brow in the brow for, man yet for all fearude, browude brow as for of, fear him bright eye in yellow in fairly fear will was fear over fear are fear there in. to for fear fear for fear his's the and and, fear to to fear of by by the Mr to Utterson regarded's fear that which fear, and fear the them a bright fear, my of fear be fear fear by they fear
Recorded frames: 128
Saved GIF: jl_fs/llm_vs_diffusion_text/LLM_Books/diffusion_gifs/final_inference.gif
